# Budgerigar：层级整体理解与单调复读指针

事件带保存有序细节；后层 token 是持续修订的整句理解，不对应单帧或单词。连续读写时钟和单调指针替代程序化行为状态。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
print(subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())

In [ ]:
#@title 2. Drive 与特征
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
STATS_PATH=WORK_ROOT/'features'/f'stats.smoke64.arctic_slt.{FEATURE_FINGERPRINT}.pt'
SOURCE=WORK_ROOT/'checkpoints'/f'monotonic_understanding_arctic_slt_{FEATURE_FINGERPRINT}'/'best.pt'
for p in (FEATURE_MANIFEST,STATS_PATH,SOURCE): assert p.is_file(),p
import torch,json
stats=torch.load(STATS_PATH,map_location='cpu',weights_only=True)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
#@title 3. 递归层间注意力与单调性检查
from budgerigar.monotonic_echo import MonotonicEchoConfig,create_monotonic_echo
config=MonotonicEchoConfig(hidden_dim=192,event_slots=160,understanding_layers=4,update_stride=4,recursive_layer_attention=True,attention_heads=4)
model=create_monotonic_echo(config)
dummy=torch.randn(1,64,102)
with torch.no_grad(): result=model(dummy)
d=result[3]
assert torch.all(d['write_phase'][:,1:]>=d['write_phase'][:,:-1])
assert torch.all(d['read_phase'][:,1:]>=d['read_phase'][:,:-1])
assert len(model.layer_attention)==config.understanding_layers
print(f'parameters={sum(p.numel() for p in model.parameters())/1e6:.2f}M',result[0].shape,d['understanding'].shape)

In [ ]:
#@title 4. T4 递归注意力 smoke training
MAX_STEPS=200 #@param {type:'integer'}
BATCH_SIZE=2 #@param {type:'integer'}
from budgerigar.train_dual_path import DualPathTrainingConfig,train_dual_path_echo
RUN_DIR=WORK_ROOT/'checkpoints'/f'monotonic_recursive_attention_arctic_slt_{FEATURE_FINGERPRINT}'
training=DualPathTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,clock_weight=1.0,initialization_checkpoint=str(SOURCE))
report=train_dual_path_echo(FEATURE_MANIFEST,RUN_DIR,training,config,stats,model_factory=create_monotonic_echo,architecture='monotonic_understanding_echo',ablation_names=('events','understanding'))
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 联合评估
from budgerigar.evaluate_echo import evaluate_checkpoint
from budgerigar.evaluate_content import evaluate_content
checkpoint=RUN_DIR/'best.pt'; out=RUN_DIR/'joint_evaluation'
behavior=evaluate_checkpoint(checkpoint,FEATURE_MANIFEST,out,max_pairs=32)
content=evaluate_content(checkpoint,FEATURE_MANIFEST,out,max_pairs=32,candidates=16)
best=min(report['history'],key=lambda x:x['validation_repeat_l1'])
print('content:',json.dumps(content,ensure_ascii=False,indent=2))
print(json.dumps({'behavior_pass':behavior['behavior_pass'],'content_pass':content['content_pass'],'best_ablation':best},ensure_ascii=False,indent=2))